# 第09課 - 元認知設計模式


## 設定

此筆記本示範如何使用 Microsoft Agent Framework 的 Metacognition 設計模式。

**先決條件：**
- 透過環境變數設定的 Azure OpenAI 部署
- 已使用 Azure CLI 登入 (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 什麼是後設認知？

後設認知是 <strong>思考自己的思考過程</strong>。在 AI 代理的背景下，它的意思是建立能夠：

- <strong>自我反思</strong> 它們自己的輸出和推理過程
- <strong>偵測錯誤</strong> 並且能優雅地恢復，而不是悄無聲息地失敗
- <strong>評估</strong> 它們的回應是否完整且有幫助
- <strong>調整</strong> 它們的策略，當初始方法失效時（例如，退回備用系統）

一個具有後設認知的代理不只是回答問題 — 它會監控自己的表現並即時調整。


## 主要及備用工具

一個常見的後設認知模式是 <strong>備援策略</strong>。代理人會先嘗試使用主要工具；如果失敗（例如，發生 404 錯誤），代理人會識別此失敗並透明地切換到備用工具。

這與現實世界中的系統相似，主要服務可能無法使用，代理人必須自行診斷問題後再選擇替代方案。

以下我們定義兩個航班查詢工具：
- <strong>主要</strong> — 覆蓋巴黎、東京及巴塞隆拿
- <strong>備用</strong> — 覆蓋柏林、悉尼及紐約市


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## 具備錯誤恢復功能的自我反思代理

以下代理被指示先嘗試主要飛行系統，識別故障，並透明地回退到備用系統。每次回應後，它會簡短自我評估是否完全回答了用戶的問題。


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## 自我評估模式

元認知的另一個面向是<strong>自我評估</strong>：一個獨立的代理人（或同一代理人在第二次審查中）對回應的完整性、準確性和有用性進行檢視。

以下我們建立一個 `ResponseEvaluator` 代理人，針對三個維度評分旅遊代理人的回應。


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Summary

In this lesson you learned how to build **metacognitive agents** using the Microsoft Agent Framework:

- **Self-reflection**: Agents that monitor their own reasoning and transparently communicate what happened.
- **Error recovery with fallbacks**: A primary + backup tool pattern where the agent detects failures (e.g., 404 errors) and automatically tries an alternative source.
- **Self-evaluation**: A separate evaluator agent that scores responses for completeness, accuracy, and helpfulness.

These patterns make agents more robust, transparent, and trustworthy — critical qualities for production deployments.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
